# Phase 4 — does it admit when it doesn't know?

The project is called *hallucination-free*. AyaTEC contains **33 questions the
Qur'an does not answer** — "Who was the Queen of Sheba?", "Where is the first
qibla?" — labelled `zero_answer` with empty `verse_keys`.

A system that invents a verse for those is hallucinating. Nobody has measured it.

Three conditions, so the guardrail's contribution is isolated:

| condition | what it adds |
|---|---|
| naive | RAG with no abstention instruction |
| abstain | told explicitly to answer INSUFFICIENT when the verses don't cover it |
| gate | the above, plus refusing before the LLM when top similarity < 0.62 |

Retrieval is the recommended config from the last report: **base GATE-AraBert-v1,
verses only**.

**T4 GPU → Run all.** ~15 minutes, mostly waiting on the free Groq tier.

### 1. Setup

In [ ]:
# ---- Cell 1: setup ----
import os, sys, shutil, subprocess, json, time, math, re, random
import numpy as np, statistics as st
import torch
from google.colab import drive
print("CUDA:", torch.cuda.is_available())
if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")

ROOT="/content/drive/MyDrive"
P1FIX=f"{ROOT}/Phase1_Project/data_fix_output"
ROMA=f"{ROOT}/Phase2_Project/Roma_output"
OUT=f"{ROOT}/Phase4_Project"; os.makedirs(f"{OUT}/data", exist_ok=True)
PROJECT="/content/QuranicRAG"
os.makedirs(f"{PROJECT}/quranNLP/shared/data", exist_ok=True); os.chdir(PROJECT)

shutil.rmtree("/content/_repo", ignore_errors=True)
subprocess.run(["git","clone","--depth","1",
 "https://github.com/Laiba-Noor/quranic-rag-hallucination-free.git","/content/_repo"],check=True)
CSV=f"{PROJECT}/quranNLP/shared/data/final_cross_reference_index.csv"
if not os.path.exists(CSV):
    shutil.copy(f"{P1FIX}/shared_data/final_cross_reference_index.csv", CSV)
try:
    import hnswlib, sentence_transformers, groq   # noqa
    print("deps present")
except ImportError:
    !pip install -q sentence-transformers hnswlib groq
print("SETUP OK")

### 2. Groq

In [ ]:
# ---- Cell 2: Groq ----
from groq import Groq

# Key is inline so the notebook runs with no setup.
# Do NOT commit this file to GitHub - a public commit gets the key auto-revoked.
GROQ_API_KEY = "YOUR_GROQ_API_KEY"
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY") or GROQ_API_KEY
except Exception:
    pass
client = Groq(api_key=GROQ_API_KEY)

# Groq retires models without notice - llama-3.3-70b-versatile, which every
# notebook in this project hardcodes, no longer exists. So ask, don't assume.
available = sorted(m.id for m in client.models.list().data)
SKIP = ("whisper", "tts", "guard", "embed", "orpheus", "safeguard")
chat = [m for m in available if not any(s in m.lower() for s in SKIP)]
print("chat models on your account:")
for m in chat:
    print("  ", m)

# allam-2-7b is Arabic-native and emits the required format cleanly.
# gpt-oss-120b is stronger but spends ~400 tokens reasoning before it answers.
PREFERENCE = ["allam-2-7b", "openai/gpt-oss-120b", "qwen/qwen3.6-27b", "openai/gpt-oss-20b"]
MODEL = next((m for m in PREFERENCE if m in chat), chat[0] if chat else None)
if MODEL is None:
    raise SystemExit("no usable chat model on this account")

# reasoning models must be given room for the reasoning AND the answer
MAX_TOK = 900 if "gpt-oss" in MODEL or "qwen3" in MODEL else 420

r = client.chat.completions.create(model=MODEL, max_tokens=MAX_TOK, temperature=0,
        messages=[{"role": "user", "content": "Reply with exactly: OK"}])
print(f"\nusing {MODEL}  (max_tokens={MAX_TOK})")
print("reachable:", (r.choices[0].message.content or "").strip()[:40])

### 3. Retrieval

In [ ]:
# ---- Cell 3: retrieval, base model, verses-only (the recommended config) ----
import csv; csv.field_size_limit(sys.maxsize)
from sentence_transformers import SentenceTransformer
import hnswlib

rows=list(csv.DictReader(open(CSV, encoding="utf-8")))
VERSES=[(r["verse_key"], (r.get("clean_verse") or "").strip())
        for r in rows if (r.get("clean_verse") or "").strip()]
TEXT={k:t for k,t in VERSES}; KEYS=[k for k,_ in VERSES]
print(f"{len(VERSES)} verses")

model=SentenceTransformer("Omartificial-Intelligence-Space/GATE-AraBert-v1")
model.max_seq_length=64
emb=model.encode([t for _,t in VERSES], convert_to_numpy=True, normalize_embeddings=True,
                 show_progress_bar=True, batch_size=256).astype(np.float32)
ix=hnswlib.Index(space="cosine", dim=emb.shape[1])
ix.init_index(max_elements=len(VERSES), ef_construction=200, M=16)
ix.add_items(emb, ids=np.arange(len(VERSES))); ix.set_ef(128)

def retrieve(q, k=8):
    lab,dist=ix.knn_query(model.encode([q], convert_to_numpy=True,
                                       normalize_embeddings=True), k=k)
    return [(KEYS[i], TEXT[KEYS[i]], float(1-d)) for i,d in zip(lab[0], dist[0])]

demo=retrieve("ما حكم الربا في الإسلام", 3)
for vk,t,s in demo: print(f"  [{vk}] {s:.3f}  {t[:52]}")

### 4. Questions

In [ ]:
# ---- Cell 4: the two question sets ----
def fetch(name,*c):
    dst=f"{OUT}/data/{name}"
    if os.path.exists(dst): return dst
    for p in c:
        if os.path.exists(p): shutil.copy(p,dst); return dst
    from google.colab import files
    print("Upload",name); up=files.upload(); shutil.copy(list(up.keys())[0],dst); return dst

aya=json.load(open(fetch("ayatec_records.json",
    "/content/_repo/Data/ayatec_records.json",
    f"{ROMA}/data/ayatec_records.json"), encoding="utf-8"))

VALID=set(TEXT)
ANSWERABLE=[(r["question"], {v for v in r["verse_keys"] if v in VALID})
            for r in aya if r.get("question") and r.get("verse_keys")]
ANSWERABLE=[(q,g) for q,g in ANSWERABLE if g]
UNANSWERABLE=[r["question"] for r in aya
              if r.get("question") and r.get("question_type")=="zero_answer"]

print(f"answerable   : {len(ANSWERABLE)}")
print(f"unanswerable : {len(UNANSWERABLE)}   <- the abstention test")
print("\nexamples of questions the Qur'an does NOT answer:")
for q in UNANSWERABLE[:4]: print("   ", q)

N_ANSWERABLE=60          # raise for a fuller run; each costs 3 LLM calls
random.seed(42)
SAMPLE=random.sample(ANSWERABLE, min(N_ANSWERABLE, len(ANSWERABLE)))
print(f"\nusing {len(SAMPLE)} answerable + {len(UNANSWERABLE)} unanswerable "
      f"= {(len(SAMPLE)+len(UNANSWERABLE))*3} LLM calls")

### 5. Prompts and parsing

In [ ]:
# ---- Cell 5: three conditions ----
def context_block(hits):
    return "\n".join(f"[{vk}] {t}" for vk,t,_ in hits)

BASE_RULES=("أنت مساعد يجيب على الأسئلة بالاعتماد على آيات القرآن الكريم المعطاة فقط.\n"
            "اذكر كل آية تستشهد بها بالصيغة [رقم السورة:رقم الآية].")

FORMAT=("\n\nأجب بهذا الشكل بالضبط:\n"
        "VERDICT: ANSWER أو INSUFFICIENT\n"
        "CITATIONS: [x:y], [x:y]\n"
        "ANSWER: <إجابتك>")

def prompt_naive(q, hits):
    return (BASE_RULES + FORMAT + "\n\nالآيات:\n" + context_block(hits) +
            f"\n\nالسؤال: {q}")

def prompt_abstain(q, hits):
    return (BASE_RULES +
            "\nإذا لم تكن الآيات المعطاة كافية للإجابة، اكتب INSUFFICIENT ولا تخمّن." +
            FORMAT + "\n\nالآيات:\n" + context_block(hits) + f"\n\nالسؤال: {q}")

SIM_GATE=0.62            # abstain before calling the LLM if retrieval is weak

def call_llm(prompt, retries=5):
    for a in range(retries):
        try:
            r=client.chat.completions.create(model=MODEL, temperature=0, max_tokens=MAX_TOK,
                messages=[{"role":"user","content":prompt}])
            return r.choices[0].message.content
        except Exception as e:
            if a==retries-1: return f"VERDICT: ERROR\nCITATIONS:\nANSWER: {e}"
            time.sleep(2*(a+1))

CITE=re.compile(r"\[?\s*(\d{1,3})\s*:\s*(\d{1,3})\s*\]?")

def parse(out):
    verdict="ANSWER"
    m=re.search(r"VERDICT:\s*(\w+)", out or "")
    if m: verdict=m.group(1).upper()
    cline=re.search(r"CITATIONS:(.*)", out or "")
    cites={f"{a}:{b}" for a,b in CITE.findall(cline.group(1) if cline else "")}
    return verdict, cites

# self-test the parser before spending any tokens
t="VERDICT: INSUFFICIENT\nCITATIONS: [2:275], [ 9:34 ]\nANSWER: لا يوجد"
assert parse(t)==("INSUFFICIENT", {"2:275","9:34"}), parse(t)
assert parse("VERDICT: ANSWER\nCITATIONS:\nANSWER: x")==("ANSWER", set())
print("parser OK")

### 6. Run

In [ ]:
# ---- Cell 6: run all three conditions ----
def run(question, gold, condition):
    hits=retrieve(question, 8)
    top=hits[0][2] if hits else 0.0
    retrieved={vk for vk,_,_ in hits}

    if condition=="gate" and top < SIM_GATE:
        return {"abstained":True, "cites":set(), "retrieved":retrieved,
                "top":top, "gated":True}

    p = prompt_naive(question,hits) if condition=="naive" else prompt_abstain(question,hits)
    verdict, cites = parse(call_llm(p))
    return {"abstained": verdict=="INSUFFICIENT", "cites":cites,
            "retrieved":retrieved, "top":top, "gated":False}

CONDITIONS=[("naive","no abstention instruction"),
            ("abstain","abstention instruction"),
            ("gate","instruction + similarity gate")]

RESULTS={}
for cond,label in CONDITIONS:
    rec={"unans":[], "ans":[]}
    t0=time.time()
    for q in UNANSWERABLE:
        rec["unans"].append(run(q, set(), cond))
    for q,gold in SAMPLE:
        r=run(q, gold, cond); r["gold"]=gold; rec["ans"].append(r)
    RESULTS[cond]=rec
    print(f"{label:<34} done in {time.time()-t0:>5.0f}s")

### 7. Results

In [ ]:
# ---- Cell 7: results ----
def score(rec):
    U,A=rec["unans"], rec["ans"]
    correct_refusal = sum(1 for r in U if r["abstained"])/len(U)
    # a fabricated citation is one the retriever never showed the model
    halluc = sum(1 for r in U if not r["abstained"] and (r["cites"]-r["retrieved"]))
    answered_unans = sum(1 for r in U if not r["abstained"])
    over_refusal = sum(1 for r in A if r["abstained"])/len(A)
    answered=[r for r in A if not r["abstained"]]
    grounded = sum(1 for r in answered if r["cites"] and not (r["cites"]-r["retrieved"]))
    correct  = sum(1 for r in answered if r["cites"] & r["gold"])
    return {
      "Correct refusal (33 unanswerable)": correct_refusal,
      "Answered anyway":                   answered_unans/len(U),
      "  ...with a fabricated citation":   halluc/len(U),
      "Over-refusal (answerable)":         over_refusal,
      "Cited a gold verse":                correct/max(len(answered),1),
      "All citations grounded in context": grounded/max(len(answered),1),
    }

rowsout={c:score(RESULTS[c]) for c,_ in CONDITIONS}
names=list(next(iter(rowsout.values())).keys())
print(f"{'metric':<38}" + "".join(f"{c:>12}" for c,_ in CONDITIONS))
print("-"*(38+12*len(CONDITIONS)))
for n in names:
    print(f"{n:<38}" + "".join(f"{rowsout[c][n]:>12.3f}" for c,_ in CONDITIONS))

print("\n" + "="*66)
best=max(CONDITIONS, key=lambda c: rowsout[c[0]]["Correct refusal (33 unanswerable)"])
print(f"best abstention: {best[1]} "
      f"({rowsout[best[0]]['Correct refusal (33 unanswerable)']:.1%} of unanswerable refused)")

json.dump({c:rowsout[c] for c,_ in CONDITIONS},
          open(f"{OUT}/phase4_abstention.json","w"), indent=2, ensure_ascii=False)
print("saved:", f"{OUT}/phase4_abstention.json")

### 8. Inspect the failures

The cases where it answered a question with no answer — and what it cited.

In [ ]:
# ---- Cell 8: read the failures ----
rec=RESULTS["abstain"]
bad=[r for r in rec["unans"] if not r["abstained"]]
print(f"{len(bad)}/{len(rec['unans'])} unanswerable questions got an answer anyway\n")
for q,r in list(zip(UNANSWERABLE, rec["unans"]))[:40]:
    if r["abstained"]: continue
    fab = r["cites"] - r["retrieved"]
    print(f"Q: {q}")
    print(f"   cited {sorted(r['cites']) or '-'} | top-sim {r['top']:.3f}"
          + (f" | FABRICATED {sorted(fab)}" if fab else ""))